This is auto loader job to load employee data from csv file from databricks volume to delta lake table in databricks. This job is scheduled to run every day at 12:00 PM.

In [2]:
print("Lets Begin with Execution of data loader notebook")

Lets Begin with Execution of data loader notebook


In [ ]:
main_catalog="workspace"
schema_name="works_dev"
volume_name="files"

In [ ]:
# 1. Define Volume and Metadata paths
# Replace with your actual catalog, schema, and volume names
volume_path = "/Volumes/main_catalog/hr_schema/employee_volume/raw_data/"
schema_location = "/Volumes/main_catalog/hr_schema/employee_volume/metadata/schema/"
checkpoint_location = "/Volumes/main_catalog/hr_schema/employee_volume/metadata/checkpoint/"

In [ ]:
# 2. Read Employee CSVs with Auto Loader
# 'rescue' mode captures malformed rows or unexpected columns in _rescued_data
df_employees = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "rescue") 
    .option("header", "true")              # Use the first line as headers
    .option("inferSchema", "true")         # Automatically detect data types (int, string, etc.)
    .load(volume_path))

In [ ]:
# 3. Write to a Delta Table in Unity Catalog
query = (df_employees.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_location)
    .option("mergeSchema", "true")          # Allow the target table schema to grow
    .trigger(availableNow=True)            # Process all new data and then stop
    .toTable("main_catalog.hr_schema.employees"))

ModuleNotFoundError: No module named 'pyspark'